# Fingerprint Enhancement & Minutiae Detection – Interactive Lab

All tunable pipeline parameters are exposed as sliders. Every slider change re-runs the **full pipeline** from that step onwards and redraws all intermediate results simultaneously, so you can see the step-by-step effect live.

**Run all cells top-to-bottom before touching any slider.**

| Grid row | Contents |
|----------|----------|
| 0 | Steps 2 & 4 – equalised image, Sobel Gx/Gy, gradient orientation |
| 1 | Steps 5 & 6 – block orientation map, histogram, sample Gabor kernel, max Gabor response |
| 2 | Steps 7 & 8 – enhanced thresh, original thresh, enhanced minutiae, original minutiae |

In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

im_glass_raw  = cv2.imread('glass.png')
im_reddit_raw = cv2.imread('reddit.jpeg')

if im_glass_raw is None:
    raise FileNotFoundError('glass.png not found.')
if im_reddit_raw is None:
    raise FileNotFoundError('reddit.jpeg not found.')

print('glass.png   :', im_glass_raw.shape)
print('reddit.jpeg :', im_reddit_raw.shape)

glass.png   : (838, 1280, 3)
reddit.jpeg : (4000, 2252, 3)


In [2]:
# =============================================================================
# Helper functions – shared by both interactive pipelines below
# =============================================================================

def ensure_odd(v):
    v = int(v)
    return v if v % 2 == 1 else v + 1


def my_conv(image, kernel, stride=1):
    '''Step 3 – strided 2-D correlation (no flip, no padding).'''
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh = (ih - kh) // stride + 1
    ow = (iw - kw) // stride + 1
    out = np.zeros((oh, ow), dtype=np.float64)
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(
                image[i*stride : i*stride+kh, j*stride : j*stride+kw] * kernel
            )
    return out


def circular_block_average(theta, d):
    '''Vectorised circular mean of orientation in non-overlapping d x d blocks.'''
    h, w = theta.shape
    ht, wt = (h // d) * d, (w // d) * d
    if ht == 0 or wt == 0:
        return np.zeros((1, 1))
    t = theta[:ht, :wt]
    c = np.cos(t).reshape(ht//d, d, wt//d, d).mean(axis=(1, 3))
    s = np.sin(t).reshape(ht//d, d, wt//d, d).mean(axis=(1, 3))
    return np.arctan2(s, c)


def make_gabor(theta, ksize, sigma, freq):
    '''Step 6 – normalised Gabor kernel at orientation theta.'''
    kern = cv2.getGaborKernel(
        (ksize, ksize), sigma, float(theta), 10.0, freq, 0, ktype=cv2.CV_64F
    )
    s = np.sum(np.abs(kern))
    return kern / s if s > 0 else kern


def detect_corners(img_u8, max_corners, quality, min_dist, blk):
    '''Step 8 – Shi-Tomasi corner detection as minutiae proxy.'''
    pts = cv2.goodFeaturesToTrack(
        img_u8,
        maxCorners=int(max_corners),
        qualityLevel=float(quality),
        minDistance=float(min_dist),
        blockSize=int(blk),
        useHarrisDetector=False,
    )
    if pts is None:
        return np.empty((0, 2), dtype=np.int32)
    return np.round(pts[:, 0, :]).astype(np.int32)


def draw_minutiae(base_gray, pts, color_bgr=(60, 60, 255), radius=4):
    vis = cv2.cvtColor(base_gray, cv2.COLOR_GRAY2BGR)
    for x, y in pts:
        cv2.circle(vis, (int(x), int(y)), radius, color_bgr, -1)
    return cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)


def run_pipeline(gray, p):
    '''Full fingerprint pipeline Steps 2-8. Returns all intermediate images.'''

    # Step 2 – histogram equalisation
    eq = cv2.equalizeHist(gray)

    # Step 4 – Gaussian blur + Sobel gradients
    gk = ensure_odd(p['gauss_k'])
    blur = cv2.GaussianBlur(eq, (gk, gk), sigmaX=p['gauss_sig'], sigmaY=p['gauss_sig'])
    sk = ensure_odd(p['sobel_k'])
    gx = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=sk)
    gy = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=sk)
    theta = np.arctan2(gy, gx)

    # Step 5 – block orientation + histogram
    d = max(2, int(p['block_d']))
    k = max(4, int(p['k_bins']))
    theta_blk = circular_block_average(theta, d)
    hist, edges = np.histogram(theta_blk.flatten(), bins=k, range=(-np.pi, np.pi))
    centers = 0.5 * (edges[:-1] + edges[1:])

    # Step 6 – Gabor filter bank built from orientation histogram
    max_o = max(1, int(p['max_orient']))
    sel_idx = np.where(hist > 0)[0]
    if len(sel_idx) == 0:
        sel_theta = np.linspace(-np.pi, np.pi, max_o, endpoint=False)
    else:
        sel_theta = centers[sel_idx]
    if len(sel_theta) > max_o:
        top = np.argsort(hist[sel_idx])[-max_o:]
        sel_theta = np.sort(sel_theta[top])

    gabor_k = ensure_odd(p['gabor_k'])
    responses, sample_kern = [], None
    src = eq.astype(np.float64)
    for th in sel_theta:
        kern = make_gabor(th, gabor_k, p['gabor_sig'], p['gabor_freq'])
        if sample_kern is None:
            sample_kern = kern
        responses.append(cv2.filter2D(src, cv2.CV_64F, kern))

    # Step 7 – max-combine Gabor responses + threshold
    enh_raw  = np.max(np.stack(responses), axis=0)
    enh_norm = cv2.normalize(enh_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, enh_bin  = cv2.threshold(enh_norm, int(p['enh_thresh']),  255, cv2.THRESH_BINARY)
    _, orig_bin = cv2.threshold(eq,       int(p['orig_thresh']), 255, cv2.THRESH_BINARY)

    # Step 8 – Shi-Tomasi corner detection as minutiae proxy
    bk = ensure_odd(p['corner_blk'])
    pts_enh  = detect_corners(enh_bin, p['max_corners'], p['quality'], p['min_dist'], bk)
    pts_orig = detect_corners(eq,      p['max_corners'], p['quality'], p['min_dist'], bk)
    vis_enh  = draw_minutiae(enh_bin, pts_enh,  color_bgr=(255, 80,  80))
    vis_orig = draw_minutiae(eq,      pts_orig, color_bgr=(80,  200, 80))

    return dict(
        eq=eq, gx=gx, gy=gy, theta=theta,
        theta_blk=theta_blk, hist=hist, edges=edges, centers=centers,
        sample_kern=sample_kern, enh_norm=enh_norm,
        enh_bin=enh_bin, orig_bin=orig_bin,
        vis_enh=vis_enh, vis_orig=vis_orig,
        pts_enh=pts_enh, pts_orig=pts_orig,
        n_enh=len(pts_enh), n_orig=len(pts_orig),
    )


def show_pipeline(r, title=''):
    '''Display all pipeline stages in a 3x4 grid.
    Row 0 = Steps 2 & 4   Row 1 = Steps 5 & 6   Row 2 = Steps 7 & 8
    '''
    fig, ax = plt.subplots(3, 4, figsize=(20, 14))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # ── Row 0: Steps 2 & 4 ────────────────────────────────────────────────
    ax[0, 0].imshow(r['eq'],    cmap='gray')
    ax[0, 0].set_title('Step 2: Hist-Equalised')
    ax[0, 1].imshow(r['gx'],    cmap='gray')
    ax[0, 1].set_title('Step 4: Sobel Gx')
    ax[0, 2].imshow(r['gy'],    cmap='gray')
    ax[0, 2].set_title('Step 4: Sobel Gy')
    ax[0, 3].imshow(r['theta'], cmap='hsv', vmin=-np.pi, vmax=np.pi)
    ax[0, 3].set_title('Step 4: Gradient Orientation')

    # ── Row 1: Steps 5 & 6 ────────────────────────────────────────────────
    ax[1, 0].imshow(r['theta_blk'], cmap='hsv', vmin=-np.pi, vmax=np.pi)
    ax[1, 0].set_title('Step 5: Block Orientation Map')

    bar_w = r['edges'][1] - r['edges'][0]
    ax[1, 1].bar(r['centers'], r['hist'], width=bar_w, edgecolor='k', color='steelblue')
    ax[1, 1].set_title('Step 5: Orientation Histogram')
    ax[1, 1].set_xlim([-np.pi, np.pi])
    ax[1, 1].grid(alpha=0.3)

    if r['sample_kern'] is not None:
        ax[1, 2].imshow(r['sample_kern'], cmap='viridis')
        ax[1, 2].set_title('Step 6: Sample Gabor Kernel')
    ax[1, 3].imshow(r['enh_norm'], cmap='gray')
    ax[1, 3].set_title('Step 6-7: Max Gabor Response')

    # ── Row 2: Steps 7 & 8 ────────────────────────────────────────────────
    ax[2, 0].imshow(r['enh_bin'],  cmap='gray')
    ax[2, 0].set_title('Step 7: Enhanced Thresholded')
    ax[2, 1].imshow(r['orig_bin'], cmap='gray')
    ax[2, 1].set_title('Step 7: Original Thresholded')
    n_enh  = r['n_enh']
    n_orig = r['n_orig']
    ax[2, 2].imshow(r['vis_enh'])
    ax[2, 2].set_title('Step 8: Enhanced Minutiae (' + str(n_enh) + ' pts)')
    ax[2, 3].imshow(r['vis_orig'])
    ax[2, 3].set_title('Step 8: Original Minutiae (' + str(n_orig) + ' pts)')

    for a in ax.flat:
        a.axis('off')
    ax[1, 1].axis('on')   # histogram needs visible axes

    plt.tight_layout()
    plt.show()


print('Pipeline helpers ready.')

Pipeline helpers ready.


## Steps 1–8: `glass.png` – Full Interactive Pipeline

Step 1 crop is fixed to the lab-specification region `im[80:750, 580:1100]`.  
All remaining tunable parameters are on sliders below.

In [3]:
# =============================================================================
# Steps 1–8: glass.png – interactive pipeline
# =============================================================================
GLASS_LAST = {}

# Step 1 – fixed crop as per lab instructions
im_glass = cv2.cvtColor(im_glass_raw[80:750, 580:1100, [2, 1, 0]], cv2.COLOR_BGR2GRAY)

_s = {'description_width': 'initial'}

# ── Step 4 sliders ──────────────────────────────────────────────────────────
w_gauss_k   = widgets.IntSlider(   value=7,    min=3,    max=21,   step=2,    description='Gauss k',     style=_s)
w_gauss_sig = widgets.FloatSlider( value=1.4,  min=0.3,  max=5.0,  step=0.1,  description='Gauss sigma', style=_s)
w_sobel_k   = widgets.IntSlider(   value=3,    min=3,    max=7,    step=2,    description='Sobel k',     style=_s)

# ── Step 5 sliders ──────────────────────────────────────────────────────────
w_block_d    = widgets.IntSlider(  value=12,   min=4,    max=32,   step=2,    description='Block d',     style=_s)
w_k_bins     = widgets.IntSlider(  value=12,   min=4,    max=24,   step=1,    description='Hist bins k', style=_s)
w_max_orient = widgets.IntSlider(  value=12,   min=2,    max=20,   step=1,    description='Max orient',  style=_s)

# ── Step 6 sliders ──────────────────────────────────────────────────────────
w_gabor_k    = widgets.IntSlider(  value=11,   min=7,    max=41,   step=2,    description='Gabor k',     style=_s)
w_gabor_sig  = widgets.FloatSlider(value=3.0,  min=0.5,  max=10.0, step=0.1,  description='Gabor sigma', style=_s)
w_gabor_freq = widgets.FloatSlider(value=0.3,  min=0.05, max=0.9,  step=0.01, description='Gabor freq',  style=_s)

# ── Step 7 sliders ──────────────────────────────────────────────────────────
w_enh_thresh  = widgets.IntSlider( value=120,  min=0,    max=255,  step=1,    description='Enh thresh',  style=_s)
w_orig_thresh = widgets.IntSlider( value=155,  min=0,    max=255,  step=1,    description='Orig thresh', style=_s)

# ── Step 8 sliders ──────────────────────────────────────────────────────────
w_max_corners = widgets.IntSlider( value=200,  min=20,   max=800,  step=10,   description='Max corners', style=_s)
w_quality     = widgets.FloatSlider(value=0.01, min=0.001, max=0.2, step=0.001,
                                    description='Quality lvl', style=_s, readout_format='.4f')
w_min_dist    = widgets.IntSlider( value=8,    min=1,    max=50,   step=1,    description='Min dist',    style=_s)
w_corner_blk  = widgets.IntSlider( value=5,    min=3,    max=21,   step=2,    description='Corner blk',  style=_s)

glass_controls = dict(
    gauss_k=w_gauss_k,  gauss_sig=w_gauss_sig,  sobel_k=w_sobel_k,
    block_d=w_block_d,  k_bins=w_k_bins,         max_orient=w_max_orient,
    gabor_k=w_gabor_k,  gabor_sig=w_gabor_sig,   gabor_freq=w_gabor_freq,
    enh_thresh=w_enh_thresh,  orig_thresh=w_orig_thresh,
    max_corners=w_max_corners, quality=w_quality,
    min_dist=w_min_dist, corner_blk=w_corner_blk,
)

def glass_update(**kw):
    result = run_pipeline(im_glass, kw)
    GLASS_LAST.update(result)
    show_pipeline(result, title='glass.png  -  Steps 1-8 (live)')

ui_glass = widgets.VBox([
    widgets.HTML('<h4>Steps 1-8 : glass.png (Step 1 crop fixed)</h4>'),
    widgets.HTML('<b>Step 4 - Gaussian / Sobel</b>'),
    widgets.HBox([w_gauss_k, w_gauss_sig, w_sobel_k]),
    widgets.HTML('<b>Step 5 - Block Orientation</b>'),
    widgets.HBox([w_block_d, w_k_bins, w_max_orient]),
    widgets.HTML('<b>Step 6 - Gabor Filter</b>'),
    widgets.HBox([w_gabor_k, w_gabor_sig, w_gabor_freq]),
    widgets.HTML('<b>Step 7 - Thresholds</b>'),
    widgets.HBox([w_enh_thresh, w_orig_thresh]),
    widgets.HTML('<b>Step 8 - Corner / Minutiae Detection</b>'),
    widgets.HBox([w_max_corners, w_quality, w_min_dist, w_corner_blk]),
])

out_glass = widgets.interactive_output(glass_update, glass_controls)
display(ui_glass, out_glass)

Output()

In [6]:
# =============================================================================
# Step 9: reddit.jpeg – interactive pipeline (shared sliders with glass.png)
# Fixed crop: im_reddit_raw[1600:2350, 1250:1700]
# =============================================================================
REDDIT_LAST = {}

im_reddit = cv2.cvtColor(im_reddit_raw[1600:2350, 1250:1700], cv2.COLOR_BGR2GRAY)

def reddit_update(**kw):
    result = run_pipeline(im_reddit, kw)
    REDDIT_LAST.update(result)
    show_pipeline(result, title='reddit.jpeg  (crop 1600:2350, 1250:1700)  -  Steps 1-8 (live)')

ui_reddit = widgets.VBox([
    widgets.HTML('<h4>Step 9 : reddit.jpeg (fixed crop, same sliders as glass.png)</h4>'),
    widgets.HTML('<b>Step 4 - Gaussian / Sobel</b>'),
    widgets.HBox([w_gauss_k, w_gauss_sig, w_sobel_k]),
    widgets.HTML('<b>Step 5 - Block Orientation</b>'),
    widgets.HBox([w_block_d, w_k_bins, w_max_orient]),
    widgets.HTML('<b>Step 6 - Gabor Filter</b>'),
    widgets.HBox([w_gabor_k, w_gabor_sig, w_gabor_freq]),
    widgets.HTML('<b>Step 7 - Thresholds</b>'),
    widgets.HBox([w_enh_thresh, w_orig_thresh]),
    widgets.HTML('<b>Step 8 - Corner / Minutiae Detection</b>'),
    widgets.HBox([w_max_corners, w_quality, w_min_dist, w_corner_blk]),
])

out_reddit = widgets.interactive_output(reddit_update, glass_controls)
display(ui_reddit, out_reddit)

Output()

In [7]:
# =============================================================================
# Step 9 – Save the best enhanced-minutiae images with cv2.imwrite
# =============================================================================

_save_label = widgets.HTML('<i>Not saved yet.</i>')

def _do_save(_):
    saved = []
    pairs = [(GLASS_LAST, 'glass_minutiae.png'), (REDDIT_LAST, 'reddit_minutiae.png')]
    for last, fname in pairs:
        if last.get('vis_enh') is not None:
            bgr = cv2.cvtColor(last['vis_enh'], cv2.COLOR_RGB2BGR)
            cv2.imwrite(fname, bgr)
            saved.append(fname)
    if saved:
        _save_label.value = '<b>Saved: ' + ', '.join(saved) + '</b>'
    else:
        _save_label.value = '<b>Nothing to save – run both pipelines above first.</b>'

btn_save = widgets.Button(
    description='Save Best Minutiae Images',
    button_style='success',
    icon='save',
)
btn_save.on_click(_do_save)
display(widgets.HBox([btn_save, _save_label]))

In [ ]:
# =============================================================================
# Step 9 – Load saved minutiae images and display with matplotlib
# =============================================================================

_show_out = widgets.Output()

def _do_show(_):
    with _show_out:
        clear_output(wait=True)
        entries = [
            ('glass_minutiae.png',  'glass.png  -  Best Enhanced Minutiae'),
            ('reddit_minutiae.png', 'reddit.jpeg  -  Best Enhanced Minutiae'),
        ]
        imgs, labels = [], []
        for fname, lbl in entries:
            if os.path.exists(fname):
                loaded = cv2.imread(fname)
                if loaded is not None:
                    imgs.append(cv2.cvtColor(loaded, cv2.COLOR_BGR2RGB))
                    labels.append(lbl)
        if not imgs:
            print('No saved images found. Click Save above first.')
            return
        fig, axes = plt.subplots(1, len(imgs), figsize=(10 * len(imgs), 8))
        if len(imgs) == 1:
            axes = [axes]
        for axis, img, lbl in zip(axes, imgs, labels):
            axis.imshow(img)
            axis.set_title(lbl, fontsize=14)
            axis.axis('off')
        plt.suptitle('Step 9: Best Minutiae Results (loaded from disk)', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

btn_show = widgets.Button(
    description='Load and Show Saved Images',
    button_style='info',
    icon='eye',
)
btn_show.on_click(_do_show)
display(btn_show, _show_out)

Button(button_style='info', description='Load and Show Saved Images', icon='eye', style=ButtonStyle())

Output()